# 05 — Structure module (`structure_module.py`)

The structure module converts Evoformer features into one 3D position and orientation per residue. Its core operation, invariant point attention (IPA), combines learned features with distances between learned 3D points.

## Stage map

```text
single representation s + pair representation z
                       |
                       v
       identity position/orientation for every residue
                       |
             +---------v-----------------------------+
             | Invariant Point Attention             |
             | content + pair bias + 3D point distance|
             +-------------------+-------------------+
                                 v
                         transition on s
                                 |
                                 v
                 quaternion + translation update
                                 |
                                 v
                  new residue positions/orientations
                                 |
                    repeat for n_ipa iterations
                                 |
                                 v
                residue frames T + C-alpha coordinates
```

**Read alongside:** `../src/af2_from_scratch/structure_module.py`. `AlphaFold 2 from Scratch` intentionally predicts backbone/C-alpha geometry only; it does not reconstruct side-chain atoms.

In [ ]:
import sys
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, "../src")  # package source lives one level up
torch.manual_seed(0)
plt.rcParams["figure.figsize"] = (8, 4)

## 1. IPA in one picture
Per residue, IPA predicts query/key/value **points** in the *local* frame, lifts them to global space with T, and adds −γ·‖q−k‖² to the attention scores: residues whose points are close attend to each other. Outputs are read back through T⁻¹ — hence *invariant*.

In [ ]:
from af2_from_scratch import AF2Config
from af2_from_scratch.geometry import make_T
from af2_from_scratch.structure_module import IPA

cfg = AF2Config()
n_res = 59
s = torch.randn(n_res, cfg.c_s)
z = torch.randn(n_res, n_res, cfg.c_z)
T = make_T(torch.eye(3).expand(n_res, 3, 3), torch.zeros(n_res, 3))
ipa_out = IPA(cfg)(s, z, T)
print("IPA input/output:", tuple(s.shape), "->", tuple(ipa_out.shape))

## 2. The iterative structure module
`n_ipa` rounds of: IPA → transition → **backbone update** (a small quaternion+translation predicted from s, composed onto the frame). Frames start at the origin and literally *grow* the protein.

In [ ]:
from af2_from_scratch.structure_module import StructureModule

sm = StructureModule(cfg)
T_out, s_out = sm(s, z)
ca = T_out[..., :3, 3]
print("frames:", tuple(T_out.shape), "  CA:", tuple(ca.shape))
print(f"structure module params: {sum(p.numel() for p in sm.parameters()) / 1e3:.0f}k")

## 3. Inspect the geometric output

The model returns a 4×4 frame per residue. The translation column is the predicted Cα position. Kabsch RMSD compares this untrained prediction with the teacher after removing arbitrary global rotation and translation.

In [ ]:
from af2_from_scratch.geometry import kabsch_rmsd
from af2_from_scratch.teacher import load_teacher

teacher = load_teacher("../examples/tautomerase/teacher.cif", n_res=ca.shape[0])
print("untrained CA-RMSD vs teacher: %.2f A" % kabsch_rmsd(ca.detach(), teacher["CA"]))
fig = plt.figure(figsize=(10, 4))
ax = fig.add_subplot(121, projection="3d")
ax.plot(*teacher["CA"].numpy().T, "o-", ms=3, lw=1)
ax.set_title("teacher")
ax2 = fig.add_subplot(122, projection="3d")
ax2.plot(*ca.detach().numpy().T, "o-", ms=3, lw=1, color="orange")
ax2.set_title("student (untrained)")
plt.show()

**Next:** `06_full_model.ipynb` connects feature extraction, embedding, Evoformer, recycling, and structure prediction.